# Scryfall Tags: Multi-Label Classification  
__Objective:__ Since the scryfall tags are in essence a collection of multiple labels for each card, this problem is at it's core a mutli-label classification task. However, given that there are nearly as many labels as cards, it will be easier to frame this as a seq2seq problem.

## Packages and Data

In [1]:
# # UNCOMMENT if operating in google colab

# ## mount to drive
# from google.colab import drive
# drive.mount('/content/gdrive', force_remount=True)

# ## ensure current directory is identified
# import os
# PROJECT_PATH = '/content/gdrive/MyDrive/data_science/scryfall-llm-sandbox'
# os.chdir(f"{PROJECT_PATH}/notebooks")
# print(f"Current Directory: {os.getcwd()}")

# # check that GPUs are available
# import torch
# print(f'GPUs Available?: {torch.cuda.is_available()}')
# print(f"Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

In [2]:
# connect project directory
import sys
from pathlib import Path
dir = str(Path(Path.cwd()).parents[0])
if dir not in sys.path:
    sys.path.append(dir)

# ## UNCOMMENT if operating in gdrive colab
# gdir = str(Path.cwd()) +  '/MyDrive/data_science/scryfall-llm-sandbox'
# if gdir not in sys.path:
#   sys.path.append(gdir)

In [3]:
# params

## data gathering
from src.config import BUILD_DATASET, TASK, TAG_SIZE, DATASET_SIZE_N, TEST_SIZE_N
from src.config import MAX_INPUT_LENGTH, MAX_TARGET_LENGTH

## modeling
from src.config import MODEL_NAME
from src.config import BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, NUM_EPOCHS
from src.config import GENERATION_MAX_LENGTH, GENERATION_NUM_BEAMS

## save model
from src.config import OUTPUT_DIR

In [4]:
# packages

## load from project directory
from src.data_gathering.scryfall_dataset import ScryfallDataset
if TASK == 'seq2seq':
    from src.fine_tuning.fine_tune_seq2seq import FineTuneLLM
elif TASK == 'multi_label_classification':
    from src.fine_tuning.fine_tune_multi_lab import FineTuneLLM

In [5]:
# get data
sf = ScryfallDataset(task = TASK)

## build dataset as needed
if BUILD_DATASET:
    sf.build_dataset(
        tag_path = '../reports/scryfall_tags.json',
        train_size_pct = 0.8,
        truncate_dataset = DATASET_SIZE_N,
        test_size_n = TEST_SIZE_N,
        top_n_tags = TAG_SIZE
    )

## load dataset
sf.load_hf_dataset(
    train_path = f'../data/scryfall_{TASK}_train.json',
    val_path = f'../data/scryfall_{TASK}_val.json',
    test_path = f'../data/scryfall_{TASK}_test.json'
)

Scryfall Tag Question Answering Dataset Built
	Train Records = 6773
	Validation Records = 1694
	Test Records = 50
	Records saved to...
		../data/scryfall_multi_label_classification_train.json
		../data/scryfall_multi_label_classification_val.json
		../data/scryfall_multi_label_classification_test.json
	NOTE: This method does not create the huggingface dataset object. Run load_dataset() for that.
Scryfall Tag Multi Label Classification Dataset Loaded
	Train Records = 6773
	Val Records = 1694
	Test Records = 50
	Count Unique Tags = 499


## Modeling

In [6]:
# Multi Label Classification Fine Tuning
tagger = FineTuneLLM(
    model_name = MODEL_NAME,
    dataset = sf.dataset,
    n_labels = len(sf.unique_tags),
    label2id = sf.label2id,
    id2label = sf.id2label,
    class_weights = sf.class_weights
)

tagger.prepare_data(
    max_input_length = MAX_INPUT_LENGTH
)

tagger.train(
    batch_size = BATCH_SIZE,
    n_epochs = NUM_EPOCHS,
    learning_rate = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
    # output_dir = f'{PROJECT_PATH}/models/scryfall_auto_tagger' # uncomment if in google colab
    output_dir = f'../models/scryfall_auto_tagger'
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,121,779 || all params: 68,458,982 || trainable%: 1.6386


Map:   0%|          | 0/6773 [00:00<?, ? examples/s]

Map:   0%|          | 0/1694 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Using device: NVIDIA GeForce GTX 1650 with Max-Q Design


c:\Documents\GitHub\scryfall-llm-sandbox\src\fine_tuning\fine_tune_multi_lab.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.pos_weight = torch.tensor(class_weights, dtype=torch.float32)


Epoch,Training Loss,Validation Loss,Micro Precision,Micro Recall,Micro F1,Macro Precision,Macro Recall,Macro F1
1,0.392664,0.390831,0.099880,0.279084,0.147111,0.006822,0.040786,0.010686
2,0.429667,0.341547,0.116124,0.436196,0.183418,0.024988,0.103736,0.031670
3,0.281625,0.278120,0.183468,0.517750,0.270930,0.079658,0.184035,0.091549
4,0.229865,0.231882,0.208837,0.612737,0.311505,0.148235,0.329354,0.176407
5,0.191891,0.205425,0.227819,0.678100,0.341054,0.192495,0.430642,0.232023
6,0.222017,0.188216,0.231484,0.714440,0.349671,0.201290,0.495188,0.253422
7,0.159053,0.177956,0.252780,0.733389,0.375972,0.222357,0.528297,0.285591
8,0.082346,0.170195,0.275603,0.735548,0.400968,0.237383,0.552910,0.307379
9,0.211912,0.162282,0.278250,0.761334,0.407550,0.244720,0.588481,0.322584
10,0.136361,0.160086,0.296024,0.758095,0.425786,0.261675,0.590677,0.338723


c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error [Errno 11001] getaddrinfo failed - silently ignoring the lookup for the file config.json in distilbert-base-uncased.
  warnings.warn(
c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in distilbert-base-uncased - will assume that the vocabulary was not modified.
  warnings.warn(
c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error [Errno 11001] getaddrinfo failed - silently ignoring the lookup for the file config.json in distilbert-base-uncased.
  warnings.warn(
c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in distilbert-base-uncased - will assu

In [7]:
# SEQ2SEQ fine tune the model
if TASK == 'seq2seq':
    tagger = FineTuneLLM(
        model_name = MODEL_NAME,
        dataset = sf.dataset
    )
    tagger.prepare_data(
        max_input_length = MAX_INPUT_LENGTH,
        max_target_length = MAX_TARGET_LENGTH
    )
    tagger.train(
        batch_size = BATCH_SIZE,
        n_epochs = NUM_EPOCHS,
        learning_rate = LEARNING_RATE,
        weight_decay = WEIGHT_DECAY,
        generation_max_length = GENERATION_MAX_LENGTH,
        generation_num_beams = GENERATION_NUM_BEAMS,
        # output_dir = f'{PROJECT_PATH}/models/scryfall_auto_tagger' # uncomment if in google colab
        output_dir = f'../models/scryfall_auto_tagger'
    )

In [8]:
if TASK == 'seq2seq':
    # Assuming 'tagger' is your FineTuneLLM instance
    from src.utils.debug_autotagger import debug_autotagger_outputs
    debug_autotagger_outputs(tagger, sf.dataset, num_samples=10)

In [9]:
if TASK == 'multi_label_classification':
    import numpy as np
    test_ids = list(sf.dataset['test']['id'])
    sample_ids = np.random.choice(test_ids, 5, replace = False)
    for card in sf.dataset['test']:
        if card['id'] in sample_ids:
            pred = tagger.generate_tags(card_text = card['document'])
            print(f'Card ID {card["id"]}')
            print(f'\tActual Tags = {sorted(card["tags"])}')
            print(f'\tPredicted Tags = {sorted(pred)}')

Card ID 3147
	Actual Tags = ['gives tap ability', 'repeatable creature tokens']
	Predicted Tags = ['activated ability', 'gives tap ability', 'multiple bodies', 'repeatable creature tokens', 'unprinted token']
Card ID 3767
	Actual Tags = ['drawback', 'theft-creature', 'triggered ability', 'upkeep cost']
	Predicted Tags = ['activated ability', 'drawback', 'sacrifice self', 'triggered ability', 'upkeep cost']
Card ID 5910
	Actual Tags = ['multiple targets', 'removal-fight']
	Predicted Tags = ['combat trick', 'multiple targets', 'one-sided fight', 'removal-fight', 'single target instant/sorcery']
Card ID 6668
	Actual Tags = ['activated ability', 'mixed subtypes', 'power matters', 'repeatable loot', 'saboteur', 'tap fuel-creature', 'triggered ability', 'unique type line']
	Predicted Tags = ['activated ability', 'draw engine', 'pure draw', 'repeatable pure draw', 'triggered ability']
Card ID 7548
	Actual Tags = ['gives pp counters', 'modal', 'repeatable creature tokens', 'synergy-commander',

## Save To Huggingface Hub

In [10]:
# UNCOMMENT TO login to the hugging face
from huggingface_hub import notebook_login
# with open('../huggingface_token.txt', 'r') as f:
#     token = f.read()

notebook_login()

In [11]:
# UNCOMMENT TO upload to huggingface hub
from huggingface_hub import get_full_repo_name
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
repo_id = get_full_repo_name(OUTPUT_DIR)
tagger.model.push_to_hub(repo_id)
tagger.tokenizer.push_to_hub(repo_id)

README.md: 0.00B [00:00, ?B/s]

c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nccru\.cache\huggingface\hub\models--ncruickshank--scryfall-auto-tagger. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/ncruickshank/scryfall-auto-tagger/commit/44a4699eb435de6a335c4ba2340e188bc3eaca62', commit_message='Upload tokenizer', commit_description='', oid='44a4699eb435de6a335c4ba2340e188bc3eaca62', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ncruickshank/scryfall-auto-tagger', endpoint='https://huggingface.co', repo_type='model', repo_id='ncruickshank/scryfall-auto-tagger'), pr_revision=None, pr_num=None)

## Load Model From Huggingface

In [14]:
# load model and use it to test results
from src.modeling.auto_tagger_multi_lab import ScryfallTaggerFromPretrained
if TASK == 'multi_label_classification':
    import numpy as np
    tagger2 = ScryfallTaggerFromPretrained(
        base_model_name = MODEL_NAME,
        n_labels = len(sf.unique_tags),
        output_dir = OUTPUT_DIR,
        id2label = sf.id2label,
        label2id = sf.label2id
    )
    test_ids = list(sf.dataset['test']['id'])
    sample_ids = np.random.choice(test_ids, 5, replace = False)
    for card in sf.dataset['test']:
        if card['id'] in sample_ids:
            pred = tagger2.generate_tags(card_text = card['document'], threshold = 0.4)
            print(f'Card ID {card["id"]}')
            print(f'\tActual Tags = {sorted(card["tags"])}')
            print(f'\tPredicted Tags = {sorted(pred)}')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Card ID 578
	Actual Tags = ['evasion', 'gains pp counters', 'saboteur', 'triggered ability']
	Predicted Tags = ['evasion', 'gains flying', 'gains pp counters', 'triggered ability', 'unique type line']
Card ID 798
	Actual Tags = ['cheaper than mv', 'cost reducer', 'delayed trigger', 'gains haste']
	Predicted Tags = ['cheaper than mv', 'delayed trigger', 'gains haste', 'gives haste', 'intervening if clause']
Card ID 5773
	Actual Tags = ['activated ability', 'death trigger', 'gives pp counters', 'gives vigilance', 'hate-artifact', 'morbid', 'namesake-spell', 'synergy-artifact']
	Predicted Tags = ['activated ability', 'death trigger', 'gains pp counters', 'gives vigilance', 'synergy-artifact']
Card ID 7274
	Actual Tags = ['activated ability', 'aesthetic counter', 'cda-power', 'cda-toughness', 'evasion', 'synergy-snow', 'type change', 'warlord']
	Predicted Tags = ['activated ability', 'cda-power', 'cda-toughness', 'land conversion', 'repeatable crime']
Card ID 8272
	Actual Tags = ['deanimat

## Graveyard

In [13]:
# # upload the model to the huggingface hub
# from huggingface_hub import Repository
# from huggingface_hub import get_full_repo_name

# ## define the repo locally
# ## NOTE: Be sure to create OUTPUT_DIR in the hub manually first
# repo_name = get_full_repo_name(OUTPUT_DIR)
# repo = Repository(OUTPUT_DIR, clone_from = repo_name)

# ## save to hub
# tagger.save_to_huggingface_hub(
#     output_dir = OUTPUT_DIR,
#     repo = repo,
#     commit_message = f'Fine-tuned {MODEL_NAME} on scryfall tags.'
# )